In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pygame
import sys
import random
from pygame.locals import QUIT, KEYDOWN, K_ESCAPE, K_SPACE, K_UP

In [2]:
def createPipe():
    offset = window_height/3
    pipeHeight = game_images['pipeimage'][0].get_height()
    
    # generating random height of pipes
    y2 = offset + random.randrange(
      0, int(window_height - game_images['sea_level'].get_height() - 1.2 * offset))  
    pipeX = window_width + 10
    y1 = pipeHeight - y2 + offset
    pipe = [
      
        # upper Pipe
        {'x': pipeX, 'y': -y1},
      
          # lower Pipe
        {'x': pipeX, 'y': y2}  
    ]
    return pipe

In [3]:
# Checking if bird is above the sealevel.
def isGameOver(horizontal, vertical, up_pipes, down_pipes):
    if vertical > elevation - 25 or vertical < 0: 
        return True

    # Checking if bird hits the upper pipe or not
    for pipe in up_pipes:    
        pipeHeight = game_images['pipeimage'][0].get_height()
        if(vertical < pipeHeight + pipe['y'] 
           and abs(horizontal - pipe['x']) < game_images['pipeimage'][0].get_width()):
            return True
          
    # Checking if bird hits the lower pipe or not
    for pipe in down_pipes:
        if (vertical + game_images['flappybird'].get_height() > pipe['y']) and abs(horizontal - pipe['x']) < game_images['pipeimage'][0].get_width():
            return True
    return False

In [4]:
def flappygame():
    your_score = 0
    horizontal = int(window_width/5)
    vertical = int(window_width/2)
    ground = 0
    mytempheight = 100

    # Generating two pipes for blitting on window
    first_pipe = createPipe()
    second_pipe = createPipe()

    # List containing lower pipes
    down_pipes = [
        {'x': window_width+300-mytempheight,
         'y': first_pipe[1]['y']},
        {'x': window_width+300-mytempheight+(window_width/2),
         'y': second_pipe[1]['y']},
    ]

    # List Containing upper pipes 
    up_pipes = [
        {'x': window_width+300-mytempheight,
         'y': first_pipe[0]['y']},
        {'x': window_width+200-mytempheight+(window_width/2),
         'y': second_pipe[0]['y']},
    ]

    pipeVelX = -4 #pipe velocity along x

    bird_velocity_y = -9  # bird velocity
    bird_Max_Vel_Y = 10   
    bird_Min_Vel_Y = -8
    birdAccY = 1
    
     # velocity while flapping
    bird_flap_velocity = -8
    
    # It is true only when the bird is flapping
    bird_flapped = False  
    while True:
       
        # Handling the key pressing events
        for event in pygame.event.get():
            if event.type == QUIT or (event.type == KEYDOWN and event.key == K_ESCAPE):
                pygame.quit()
                sys.exit()
            if event.type == KEYDOWN and (event.key == K_SPACE or event.key == K_UP):
                if vertical > 0:
                    bird_velocity_y = bird_flap_velocity
                    bird_flapped = True

        # This function will return true if the flappybird is crashed
        game_over = isGameOver(horizontal, vertical, up_pipes, down_pipes)
        if game_over:
            # fill screen white on game over
            window.fill((255,255,255))
            pygame.display.update()
            pygame.time.delay(300)    # 1 second pause
            pygame.quit()
            sys.exit()
        # check for your_score
        playerMidPos = horizontal + game_images['flappybird'].get_width()/2
        for pipe in up_pipes:
            pipeMidPos = pipe['x'] + game_images['pipeimage'][0].get_width()/2
            if pipeMidPos <= playerMidPos < pipeMidPos + 4:
                  # Printing the score
                your_score += 1
                print(f"Your your_score is {your_score}")

        if bird_velocity_y < bird_Max_Vel_Y and not bird_flapped:
            bird_velocity_y += birdAccY

        if bird_flapped:
            bird_flapped = False
        playerHeight = game_images['flappybird'].get_height()
        vertical = vertical + min(bird_velocity_y, elevation - vertical - playerHeight)

        # move pipes to the left
        for upperPipe, lowerPipe in zip(up_pipes, down_pipes):
            upperPipe['x'] += pipeVelX
            lowerPipe['x'] += pipeVelX

        # Add a new pipe when the first is about
        # to cross the leftmost part of the screen
        if 0 < up_pipes[0]['x'] < 5:
            newpipe = createPipe()
            up_pipes.append(newpipe[0])
            down_pipes.append(newpipe[1])

        # if the pipe is out of the screen, remove it
        if up_pipes[0]['x'] < -game_images['pipeimage'][0].get_width():
            up_pipes.pop(0)
            down_pipes.pop(0)

        # Lets blit our game images now
        window.blit(game_images['background'], (0, 0))
        for upperPipe, lowerPipe in zip(up_pipes, down_pipes):
            window.blit(game_images['pipeimage'][0],
                        (upperPipe['x'], upperPipe['y']))
            window.blit(game_images['pipeimage'][1],
                        (lowerPipe['x'], lowerPipe['y']))

        window.blit(game_images['sea_level'], (ground, elevation))
        window.blit(game_images['flappybird'], (horizontal, vertical))
        
        # Fetching the digits of score.
        numbers = [int(x) for x in list(str(your_score))]
        width = 0
        
        # finding the width of score images from numbers.
        for num in numbers:
            width += game_images['scoreimages'][num].get_width()
        Xoffset = (window_width - width)/1.1
        
        # Blitting the images on the window.
        for num in numbers:
            window.blit(game_images['scoreimages'][num], (Xoffset, window_width*0.02))
            Xoffset += game_images['scoreimages'][num].get_width()
            
        # Refreshing the game window and displaying the score.
        pygame.display.update()
        
        # Set the framepersecond
        framepersecond_clock.tick(framepersecond)

In [5]:
def flappygame_generator(action=None):
    """
    A coroutine‐style generator:
    yields (frame, score) and accepts an action (0 or 1) sent in.
    If action==1, the bird flaps on that step.
    """
    your_score   = 0
    horizontal   = int(window_width / 5)
    vertical     = int(window_height / 2)
    ground       = 0
    mytempheight = 100

    # off‐screen canvas
    canvas = pygame.Surface((window_width, window_height))

    # initial pipes
    first_pipe, second_pipe = createPipe(), createPipe()
    down_pipes = [
        {'x': window_width + 300 - mytempheight,            'y': first_pipe[1]['y']},
        {'x': window_width + 300 - mytempheight + window_width//2, 'y': second_pipe[1]['y']},
    ]
    up_pipes = [
        {'x': window_width + 300 - mytempheight,            'y': first_pipe[0]['y']},
        {'x': window_width + 300 - mytempheight + window_width//2, 'y': second_pipe[0]['y']},
    ]

    pipeVelX           = -4
    bird_velocity_y    = -9
    bird_Max_Vel_Y     = 10
    birdAccY           = 1
    bird_flap_velocity = -8
    bird_flapped       = False

    while True:
        # handle quit events
        for e in pygame.event.get():
            if e.type == QUIT or (e.type == KEYDOWN and e.key == K_ESCAPE):
                canvas.fill((255,255,255))
                arr = pygame.surfarray.array3d(canvas)
                arr = np.transpose(arr, (1,0,2))
                yield arr, your_score
                return

        # apply external action
        if action == 1 and vertical > 0:
            bird_velocity_y = bird_flap_velocity
            bird_flapped   = True

        # collision check
        if isGameOver(horizontal, vertical, up_pipes, down_pipes):
            canvas.fill((255,255,255))
            arr = pygame.surfarray.array3d(canvas)
            arr = np.transpose(arr, (1,0,2))
            yield arr, your_score
            return

        # update score
        player_mid = horizontal + game_images['flappybird'].get_width() / 2
        for p in up_pipes:
            pmid = p['x'] + game_images['pipeimage'][0].get_width() / 2
            if pmid <= player_mid < pmid + abs(pipeVelX):
                your_score += 1

        # physics
        if bird_velocity_y < bird_Max_Vel_Y and not bird_flapped:
            bird_velocity_y += birdAccY
        bird_flapped = False
        bh = game_images['flappybird'].get_height()
        vertical += min(bird_velocity_y, elevation - vertical - bh)

        # move & spawn pipes
        for u, d in zip(up_pipes, down_pipes):
            u['x'] += pipeVelX
            d['x'] += pipeVelX
        if 0 < up_pipes[0]['x'] < 5:
            np0, np1 = createPipe()
            up_pipes.append(np0);  down_pipes.append(np1)
        if up_pipes[0]['x'] < -game_images['pipeimage'][0].get_width():
            up_pipes.pop(0);     down_pipes.pop(0)

        # draw off‐screen
        canvas.blit(game_images['background'], (0, 0))
        for u, d in zip(up_pipes, down_pipes):
            canvas.blit(game_images['pipeimage'][0], (u['x'], u['y']))
            canvas.blit(game_images['pipeimage'][1], (d['x'], d['y']))
        canvas.blit(game_images['sea_level'], (ground, elevation))
        canvas.blit(game_images['flappybird'], (horizontal, vertical))

        # draw score
        nums = [int(c) for c in str(your_score)]
        total_w = sum(game_images['scoreimages'][n].get_width() for n in nums)
        x0 = (window_width - total_w) / 2
        for n in nums:
            img = game_images['scoreimages'][n]
            canvas.blit(img, (x0, window_height * 0.02))
            x0 += img.get_width()

        # cap FPS
        framepersecond_clock.tick(framepersecond)

        # grab frame & yield, receive next action
        arr = pygame.surfarray.array3d(canvas)
        arr = np.transpose(arr, (1,0,2))
        action = yield arr, your_score


In [ ]:
def run_generator():
    """
    Drive flappygame_generator with real‐time key events,
    mirroring the behavior of Run().
    """
    global window, game_images, framepersecond, elevation
    global window_width, window_height,framepersecond_clock
    # game setup (same as in Run)
    window_width   = 600
    window_height  = 499
    elevation      = window_height * 0.8
    framepersecond = 32
    framepersecond_clock = pygame.time.Clock()

    pygame.init()
    window = pygame.display.set_mode((window_width, window_height))
    pygame.display.set_caption('Flappy Bird (generator)')
    clock = pygame.time.Clock()

    # load images
    base = '/home/hamlil/Desktop/Flappy_Bird_CNN/images'
    game_images = {}
    game_images['scoreimages'] = tuple(
        pygame.image.load(f'{base}/{i}.png').convert_alpha()
        for i in range(10)
    )
    game_images['flappybird'] = pygame.image.load(f'{base}/bird.png').convert_alpha()
    game_images['sea_level']  = pygame.image.load(f'{base}/base.jfif').convert_alpha()
    game_images['background'] = pygame.image.load(f'{base}/background.jpg').convert_alpha()
    pi = pygame.image.load(f'{base}/pipe.png').convert_alpha()
    game_images['pipeimage']  = (
        pygame.transform.rotate(pi, 180),
        pi
    )

    # prime the generator
    gen = flappygame_generator(action=None)
    frame, score = next(gen)

    running = True
    while running:
        action = 0
        for e in pygame.event.get():
            if e.type == QUIT or (e.type == KEYDOWN and e.key == K_ESCAPE):
                running = False
            elif e.type == KEYDOWN and e.key in (K_SPACE, K_UP):
                action = 1

        try:
            frame, score = gen.send(action)
        except StopIteration:
            break

        # blit the frame returned by generator
        surf = pygame.surfarray.make_surface(frame.transpose(1,0,2))
        window.blit(surf, (0, 0))
        pygame.display.update()
        print(f"Score: {score}")

        clock.tick(framepersecond)

    pygame.quit()

In [7]:
run_generator()

Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
Score: 0
S

In [ ]:
# Global Variables for the game
def Run():
    global window, game_images, framepersecond, elevation
    global window_width, window_height,framepersecond_clock
    window_width = 600
    window_height = 499

    window = pygame.display.set_mode((window_width, window_height))   
    elevation = window_height * 0.8
    game_images = {}      
    framepersecond = 32
    framepersecond_clock = pygame.time.Clock()

    # File paths
    background_image   = '/home/hamlil/Desktop/Flappy_Bird_CNN/images/background.jpg'
    pipeimage          = '/home/hamlil/Desktop/Flappy_Bird_CNN/images/pipe.png'
    sealevel_image     = '/home/hamlil/Desktop/Flappy_Bird_CNN/images/base.jfif'
    birdplayer_image   = '/home/hamlil/Desktop/Flappy_Bird_CNN/images/bird.png'




    pygame.init()  
    framepersecond_clock = pygame.time.Clock()
    pygame.display.set_caption('Flappy Bird Game')      

        # Load all the images
    game_images['scoreimages'] = tuple(
            pygame.image.load(f'/home/hamlil/Desktop/Flappy_Bird_CNN/images/{i}.png').convert_alpha()
            for i in range(10)
        )
    game_images['flappybird'] = pygame.image.load(birdplayer_image).convert_alpha()                  
    game_images['sea_level'] = pygame.image.load(sealevel_image).convert_alpha()
    game_images['background'] = pygame.image.load(background_image).convert_alpha()
    game_images['pipeimage'] = (
            pygame.transform.rotate(pygame.image.load(pipeimage).convert_alpha(), 180),
            pygame.image.load(pipeimage).convert_alpha()
        )

    print("WELCOME TO THE FLAPPY BIRD GAME")
    print("Press space or up to start the game")
    pygame.time.delay(2000)
    flappygame()
        # Start screen loop
    while True:
        horizontal = int(window_width / 5)
        vertical = int((window_height - game_images['flappybird'].get_height()) / 2)
        ground = 0

        while True:
            for event in pygame.event.get():
                if event.type == QUIT or (event.type == KEYDOWN and event.key == K_ESCAPE):
                    pygame.quit()
                    sys.exit()
                elif event.type == KEYDOWN and (event.key == K_SPACE or event.key == K_UP):
                    flappygame()  # Start the game

                # Draw welcome screen
            window.blit(game_images['background'], (0, 0))
            window.blit(game_images['flappybird'], (horizontal, vertical))
            window.blit(game_images['sea_level'], (ground, elevation))
            pygame.display.update()
            framepersecond_clock.tick(framepersecond)

In [6]:
Run()

WELCOME TO THE FLAPPY BIRD GAME
Press space or up to start the game
Your your_score is 1
Your your_score is 2
Your your_score is 3
Your your_score is 4


SystemExit: 

/home/hamlil/anaconda3/envs/flappy/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
def _transpose_surface(surface):
    arr = pygame.surfarray.array3d(surface)
    return np.transpose(arr, (1, 0, 2))

class FlappyBirdEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self, render_mode=None, pipe_gap=150, return_tensor=True, pipe_distance=300):
        super().__init__()
        # game constants
        self.window_width   = 600
        self.window_height  = 499
        self.elevation      = int(self.window_height * 0.8)
        self.pipe_vel_x     = -4
        self.gravity        = 1
        self.flap_vel       = -8
        self.max_vel_y      = 10
        self.pipe_gap       = pipe_gap
        self.pipe_distance  = pipe_distance
        self.return_tensor  = return_tensor
        self.next_pipe_x    = self.window_width + 10

        # action & observation spaces
        self.action_space = spaces.Discrete(2)
        low  = np.zeros((self.window_height, self.window_width, 3), dtype=np.uint8)
        high = np.full((self.window_height, self.window_width, 3), 255, dtype=np.uint8)
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.uint8)

        # pygame setup
        pygame.init()
        self.screen = pygame.display.set_mode((self.window_width, self.window_height))
        pygame.display.set_caption('Flappy Bird Gym')
        self.clock = pygame.time.Clock()
        self._load_images()

        # initial reset
        self.reset()

    def _load_images(self):
        base = '/home/hamlil/Desktop/Flappy_Bird_CNN/images'
        self.img_bg       = pygame.image.load(f'{base}/background.jpg').convert()
        self.img_base     = pygame.image.load(f'{base}/base.jfif').convert()
        self.img_bird     = pygame.image.load(f'{base}/bird.png').convert_alpha()
        pipe_img          = pygame.image.load(f'{base}/pipe.png').convert_alpha()
        self.img_pipe_top = pygame.transform.rotate(pipe_img, 180)
        self.img_pipe_bot = pipe_img
        self.pipe_w       = pipe_img.get_width()
        self.pipe_h       = pipe_img.get_height()
        self.base_y       = self.elevation

    def _create_pipe(self, x=None):
        # choose a random vertical position for the gap
        center_y = random.randint(
            int(self.pipe_gap/2 + 10),
            int(self.base_y - self.pipe_gap/2 - 10)
        )
        top_y = center_y - self.pipe_gap/2 - self.pipe_h
        bot_y = center_y + self.pipe_gap/2
        if x is None:
            x = self.next_pipe_x
        return {'top': {'x': x, 'y': top_y}, 'bot': {'x': x, 'y': bot_y}}

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        # reset bird state
        self.bird_x, self.bird_y = int(self.window_width/5), int(self.window_height/2)
        self.vel_y, self.flapped, self.score = 0, False, 0
        # reset pipes with fixed spacing
        self.next_pipe_x = self.window_width + 10
        pipe1 = self._create_pipe()
        pipe2 = self._create_pipe(x=self.next_pipe_x + self.pipe_distance)
        self.pipes = [pipe1, pipe2]
        self.next_pipe_idx = 0
        # advance next spawn
        self.next_pipe_x += self.pipe_distance * 2
        return self._get_obs()

    def step(self, action):
        # flap
        if action == 1 and self.bird_y > 0:
            self.vel_y, self.flapped = self.flap_vel, True
        # gravity
        if self.vel_y < self.max_vel_y and not self.flapped:
            self.vel_y += self.gravity
        self.flapped = False
        # update bird position
        bh = self.img_bird.get_height()
        self.bird_y = self.bird_y + min(self.vel_y,
                                        self.elevation - self.bird_y - bh)
        # move pipes
        for p in self.pipes:
            p['top']['x'] += self.pipe_vel_x
            p['bot']['x'] += self.pipe_vel_x
                # spawn new pipe when last pipe crosses threshold
        last_pipe = self.pipes[-1]
        if last_pipe['top']['x'] < self.window_width - self.pipe_distance:
            self.pipes.append(self._create_pipe())
            self.next_pipe_x += self.pipe_distance
        # remove offscreen pipe
        if self.pipes[0]['top']['x'] < -self.pipe_w:
            self.pipes.pop(0)
            self.next_pipe_idx = max(0, self.next_pipe_idx - 1)
        if self.pipes[0]['top']['x'] < -self.pipe_w:
            self.pipes.pop(0)
            self.next_pipe_idx = max(0, self.next_pipe_idx - 1)
        # check collisions
        done = self._is_game_over()
        # reward shaping
        reward = 0.01  # survival bonus
        pipe = self.pipes[self.next_pipe_idx]
        pipe_mid_x = pipe['top']['x'] + self.pipe_w/2
        bird_mid_x = self.bird_x + self.img_bird.get_width()/2
        if pipe_mid_x <= bird_mid_x < pipe_mid_x + abs(self.pipe_vel_x):
            self.score += 1
            reward += 1.0
            self.next_pipe_idx = min(self.next_pipe_idx+1, len(self.pipes)-1)
        if done:
            reward -= 1.0
        return self._get_obs(), reward, done, {'score': self.score}

    def _draw(self):
        # draw background, pipes, base, bird
        self.screen.blit(self.img_bg, (0, 0))
        for p in self.pipes:
            self.screen.blit(self.img_pipe_top, (p['top']['x'], p['top']['y']))
            self.screen.blit(self.img_pipe_bot, (p['bot']['x'], p['bot']['y']))
        self.screen.blit(self.img_base, (0, self.base_y))
        self.screen.blit(self.img_bird, (self.bird_x, self.bird_y))

    def _get_obs(self):
        # render frame offscreen
        self._draw()
        if self.return_tensor:
            return _transpose_surface(self.screen)
        else:
            pygame.display.update()
            self.clock.tick(32)
            return None

    def _is_game_over(self):
        # hit ground or ceiling
        if self.bird_y < 0 or self.bird_y > self.elevation - self.img_bird.get_height():
            return True
        # collision with pipes
        for p in self.pipes:
            # top pipe
            if (self.bird_y < p['top']['y'] + self.pipe_h and
                abs(self.bird_x - p['top']['x']) < self.pipe_w):
                return True
            # bottom pipe
            if (self.bird_y + self.img_bird.get_height() > p['bot']['y'] and
                abs(self.bird_x - p['bot']['x']) < self.pipe_w):
                return True
        return False

    def render(self, mode='human'):
        # human display
        self._draw()
        pygame.display.update()
        self.clock.tick(32)
        if mode == 'rgb_array':
            return _transpose_surface(self.screen)

    def close(self):
        pygame.quit()


In [2]:
Flappy_Bird_Run()  # Start the game   when the script is run

WELCOME TO THE FLAPPY BIRD GAME
Game will start in 3 seconds…


AttributeError: 'FlappyBirdEnv' object has no attribute '_load_images'

In [ ]:
if __name__ == "__main__":

    env = FlappyBirdEnv()
    obs = env.reset()
    assert env.observation_space.contains(obs), f"Reset obs {obs} out of bounds"

    total_reward = 0.0
    done = False
    step_count = 0
    print("Starting random‐action rollout…")
    while not done and step_count < 1000:
        action = env.action_space.sample()
        print(f"Step {step_count}: action = {action}")
        obs, reward, done, info = env.step(action)
        assert env.observation_space.contains(obs), f"Step obs {obs} out of bounds"
        total_reward += reward
        env.render()              # should pop up the pygame window
        step_count += 1

    print(f"Rollout finished in {step_count} steps, total reward = {total_reward}")
    env.close()


AttributeError: 'FlappyBirdEnv' object has no attribute '_load_images'

: 

In [23]:
import random
import collections
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
NUM_EPISODES   = 2000
BATCH_SIZE     = 16
GAMMA          = 0.99
LR             = 1e-4
EPS_START      = 1.0
EPS_END        = 0.1
EPS_DECAY_FRAMES = 200_000  # frames over which eps decays linearly
TARGET_UPDATE  = 1_000
BUFFER_SIZE    = 10_000

# Small DQN with Sigmoid
class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=8, stride=4),
            nn.Sigmoid(),
            nn.Conv2d(16,32, kernel_size=4, stride=2),
            nn.Sigmoid(),
        )
        # compute flattened feature size for 84×84 input
        def conv_out(sz,k,s): return (sz-(k-1)-1)//s+1
        w = conv_out(conv_out(84,8,4),4,2)
        h = conv_out(conv_out(84,8,4),4,2)
        lin = w*h*32
        self.fc = nn.Sequential(
            nn.Linear(lin, 256), nn.Sigmoid(),
            nn.Linear(256, n_actions)
        )

    def forward(self, x):
        # x: (B,H,W,3) uint8
        x = x.float().mean(-1, keepdim=True)            # grayscale
        x = x.permute(0,3,1,2)                          # (B,1,H,W)
        x = nn.functional.interpolate(x, size=(84,84), mode='bilinear', align_corners=False)
        x = x.repeat(1,3,1,1)                           # (B,3,84,84)
        x = x / 255.0
        x = self.conv(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

# Replay Buffer
Transition = collections.namedtuple('Transition',
    ('state','action','reward','next_state','done'))
class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = collections.deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, n):
        batch = random.sample(self.buf, n)
        return Transition(*zip(*batch))
    def __len__(self):
        return len(self.buf)

# Linear epsilon decay
def epsilon_by_frame(frame_idx):
    eps = EPS_START - (EPS_START - EPS_END) * min(frame_idx, EPS_DECAY_FRAMES) / EPS_DECAY_FRAMES
    return max(EPS_END, eps)

def train():
    # headless, tensor env
    env = FlappyBirdEnv(render_mode=None, return_tensor=True)
    n_actions = env.action_space.n

    policy_net = DQN(n_actions).to(device)
    target_net = DQN(n_actions).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    optimizer = optim.Adam(policy_net.parameters(), lr=LR)
    buffer = ReplayBuffer(BUFFER_SIZE)

    frame_idx = 0
    for ep in range(NUM_EPISODES):
        obs   = env.reset()
        state = torch.from_numpy(obs).unsqueeze(0).to(device)
        total_reward = 0.0
        done = False

        while not done:
            eps = epsilon_by_frame(frame_idx)
            frame_idx += 1

            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy_net(state).argmax(1).item()

            next_obs, reward, done, _ = env.step(action)
            next_state = torch.from_numpy(next_obs).unsqueeze(0).to(device)
            buffer.push(state, action, reward, next_state, done)

            state = next_state
            total_reward += reward

            if len(buffer) >= BATCH_SIZE:
                batch = buffer.sample(BATCH_SIZE)
                states      = torch.cat(batch.state)
                actions     = torch.tensor(batch.action, device=device).unsqueeze(1)
                rewards     = torch.tensor(batch.reward, device=device)
                next_states = torch.cat(batch.next_state)
                dones       = torch.tensor(batch.done, dtype=torch.float32, device=device)

                q_values = policy_net(states).gather(1, actions).squeeze()
                next_q   = target_net(next_states).max(1)[0]
                targets  = rewards + GAMMA * next_q * (1 - dones)

                loss = nn.MSELoss()(q_values, targets.detach())
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                del states, actions, rewards, next_states, dones, q_values, next_q, targets
                torch.cuda.empty_cache()

            if frame_idx % TARGET_UPDATE == 0:
                target_net.load_state_dict(policy_net.state_dict())

        print(f"Episode {ep:<4} Reward: {total_reward:.2f}")

    torch.save(policy_net.state_dict(), "dqn_flappy_adam.pth")

if __name__ == "__main__":
    train()


Episode 0    Reward: -0.59
Episode 1    Reward: -0.67
Episode 2    Reward: -0.62
Episode 3    Reward: -0.64
Episode 4    Reward: -0.63
Episode 5    Reward: -0.62
Episode 6    Reward: -0.63
Episode 7    Reward: -0.61
Episode 8    Reward: -0.66
Episode 9    Reward: -0.66
Episode 10   Reward: -0.66
Episode 11   Reward: -0.60
Episode 12   Reward: -0.61
Episode 13   Reward: -0.66
Episode 14   Reward: -0.63
Episode 15   Reward: -0.65
Episode 16   Reward: -0.60
Episode 17   Reward: -0.60
Episode 18   Reward: -0.62
Episode 19   Reward: -0.64
Episode 20   Reward: -0.65
Episode 21   Reward: -0.57
Episode 22   Reward: -0.62
Episode 23   Reward: -0.64
Episode 24   Reward: -0.59
Episode 25   Reward: -0.59
Episode 26   Reward: -0.66
Episode 27   Reward: -0.66
Episode 28   Reward: -0.61
Episode 29   Reward: -0.63
Episode 30   Reward: -0.68
Episode 31   Reward: -0.62
Episode 32   Reward: -0.63
Episode 33   Reward: -0.65
Episode 34   Reward: -0.61
Episode 35   Reward: -0.64
Episode 36   Reward: -0.62
E

: 